In [ ]:
import json, os, pathlib, shutil, subprocess, sys
os.environ["PYTHONUTF8"] = "1"

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "onnxruntime", "pycocotools"], check=True)

REPO = pathlib.Path("/tmp/shelfsense_repo")
shutil.rmtree(REPO, ignore_errors=True)
subprocess.run(["git", "clone", "--depth", "1", "https://github.com/ibrahimSumon47/ShelfSense-Counter.git", str(REPO)], check=True)

WORK = pathlib.Path("/kaggle/working")
PRED = WORK / "predictions"
PRED.mkdir(parents=True, exist_ok=True)
ONNX = sorted(p for p in pathlib.Path("/kaggle/input").rglob("*.onnx") if p.stat().st_size > 1_000_000)
print("onnx files:", [p.name for p in ONNX])
assert ONNX, "no ONNX file in /kaggle/input"

In [ ]:
# Thresholds come from SKU-110K validation only; the test split is scored once afterwards.
val_out = WORK / "val-sku110k.json"
cmd = [sys.executable, str(REPO / "training/eval_onnx.py"), "--onnx", *[str(p) for p in ONNX],
       "--dataset", "sku110k", "--split", "val", "--sku-dir", "/kaggle/input",
       "--input-size", "640", "--normalise", "no", "--center-fix", "none",
       "--score-thresh", "0.3", "--nms-thresh", "0.35", "--max-dets", "1000",
       "--grid", "0.15,0.2,0.25,0.3:0.35,0.45", "--out", str(val_out)]
print("$ " + " ".join(cmd))
subprocess.run(cmd, check=True)

val = json.loads(val_out.read_text())
rows = [r for m in val["models"] for r in m["threshold_sweep"]]
best = min(rows, key=lambda r: r["count_mae"])
print("BEST", json.dumps({k: best[k] for k in ("score_thresh", "nms_thresh", "map50", "count_mae")}))
(WORK / "thresh.json").write_text(json.dumps(best))

In [ ]:
test_out = WORK / "test-sku110k.json"
cmd = [sys.executable, str(REPO / "training/eval_onnx.py"), "--onnx", *[str(p) for p in ONNX],
       "--dataset", "sku110k", "--split", "test", "--sku-dir", "/kaggle/input",
       "--input-size", "640", "--normalise", "no", "--center-fix", "none",
       "--score-thresh", str(best["score_thresh"]), "--nms-thresh", str(best["nms_thresh"]),
       "--max-dets", "1000", "--out", str(test_out), "--predictions-dir", str(PRED)]
print("$ " + " ".join(cmd))
subprocess.run(cmd, check=True)

test = json.loads(test_out.read_text())
summary = {
    "thresholds": {"score": best["score_thresh"], "nms": best["nms_thresh"], "tuned_on": "sku110k val 200 images"},
    "val": [{'onnx_file': m['onnx_file'], **{k: m['primary'][k] for k in ('map50', 'map50_95', 'count_mae', 'count_mape_pct')}} for m in val["models"]],
    "test": [{'onnx_file': m['onnx_file'], **{k: m['primary'][k] for k in ('map50', 'map50_95', 'count_mae', 'count_mape_pct')}} for m in test["models"]],
}
(WORK / "summary.json").write_text(json.dumps(summary, indent=2))
print("SUMMARY " + json.dumps(summary))
if not summary["test"] or summary["test"][0]["map50"] <= 0:
    raise RuntimeError("test mAP50 is zero, the evaluation is wrong")